# L06: Classifier bake-off against an operational cost function

**Day 2.** Supervised learning with scikit-learn: the train/test contract, a cross-validated comparison of six model families, and an operating point chosen by operational cost rather than accuracy.

- Reads: checkpoints/L02_clean.csv, checkpoints/L04_split.csv
- Writes: work/L06_choice.json, work/L06_test_scores.csv

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L02_clean.csv",
          "checkpoints/L04_split.csv"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler


MODEL_NAMES = ["logistic regression", "k-nearest neighbours", "support vector machine", "decision tree",
               "random forest", "gradient boosting"]


def make_model(name, preprocessor, seed=2026):
    """A Pipeline of the given preprocessing and one of the course's bounded model families.

    Every family has fixed, small settings so it trains in seconds on a few thousand rows.

    Lab L06.
    """
    models = {
        "logistic regression": LogisticRegression(max_iter=1000),
        "k-nearest neighbours": KNeighborsClassifier(n_neighbors=25),
        "support vector machine": SVC(C=1.0, kernel="rbf", probability=True, random_state=seed),  # probabilities, so every family is read on one scale
        "decision tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=seed),
        "random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=1, random_state=seed),
        "gradient boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=seed),
    }
    return Pipeline([("prep", preprocessor), ("model", models[name])])


def cv_compare(pipelines, X, y, folds=5, seed=2026):
    """Out-of-fold scores and AUC for each named pipeline, from stratified k-fold cross-validation.

    Returns (table of AUC by model, DataFrame of out-of-fold scores, one column per model).

    Lab L06.
    """
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    scores, rows = {}, []
    for name, pipe in pipelines.items():
        method = "predict_proba" if hasattr(pipe.named_steps["model"], "predict_proba") and \
            getattr(pipe.named_steps["model"], "probability", True) else "decision_function"
        out = cross_val_predict(pipe, X, y, cv=cv, method=method)
        scores[name] = out[:, 1] if out.ndim == 2 else out
        rows.append({"model": name, "cv_auc": roc_auc_score(y, scores[name])})
    return pd.DataFrame(rows).set_index("model"), pd.DataFrame(scores, index=X.index)


def cost_curve(y_true, scores, cost_fn, cost_fp, thresholds=None):
    """Cost, accuracy and error counts at each threshold on the scores.

    Lab L06.
    """
    scores = np.asarray(scores)
    if thresholds is None:
        thresholds = np.unique(np.quantile(scores, np.linspace(0, 1, 201)))
    rows = []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        rows.append({"threshold": t, "cost": cost_fn * fn + cost_fp * fp, "accuracy": (tp + tn) / len(pred),
                     "missed": fn, "false_alarms": fp})
    return pd.DataFrame(rows)


def pick_threshold(curve, by="cost"):
    """Threshold with the lowest cost (by="cost") or the highest accuracy (by="accuracy").

    Lab L06.
    """
    row = curve.loc[curve["cost"].idxmin()] if by == "cost" else curve.loc[curve["accuracy"].idxmax()]
    return float(row["threshold"])


def confusion_at(y_true, scores, threshold):
    """Confusion counts at a threshold, as a dict: true negatives, false positives, false negatives, true positives.

    Lab L06.
    """
    pred = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


NUMERIC = ["aircraft_age_years", "total_flight_hours", "days_since_inspection", "ground_time_hours",
           "cargo_weight_klb", "temperature_c", "sortie_duration_hours", "open_major", "open_minor"]


CATEGORICAL = ["aircraft_type", "home_base", "mission_type"]


TARGET = "maint_delay"


def build_preprocessor(numeric=NUMERIC, categorical=CATEGORICAL):
    """Unfitted preprocessing: numeric median imputation with missing indicators, then scaling;
    categorical one-hot encoding that ignores unseen levels.

    Fit it on training rows only.

    Lab L04.
    """
    numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                              ("scale", StandardScaler())])
    return ColumnTransformer([
        ("num", numeric_steps, numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])

print("Helper functions ready")

## Part 1: Baseline

**Specification.** Establish what "good" has to beat: the accuracy of always predicting "no delay", and a logistic
regression at the default threshold of 0.5.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

df = pd.read_csv(input_path("L02_clean.csv"), encoding="utf-8")
df = df.merge(pd.read_csv(input_path("L04_split.csv"), encoding="utf-8"), on="mission_id", validate="one_to_one")
X_cols = NUMERIC + CATEGORICAL
train, test = df[df["split"] == "train"], df[df["split"] == "test"]
X_train, y_train = train[X_cols], train[TARGET]
X_test, y_test = test[X_cols], test[TARGET]

majority = 1 - y_train.mean()
print(f"always predicting 'no delay': accuracy {majority:.3f}")

# Out-of-fold predictions: every training row is scored by a model that never saw it.
auc_logit, oof_logit = cv_compare(
    {"logistic regression": make_model("logistic regression", build_preprocessor())},
    X_train, y_train, folds=5, seed=SEED)
cm = confusion_at(y_train, oof_logit["logistic regression"], 0.5)
accuracy = (cm["tp"] + cm["tn"]) / len(y_train)
print(f"logistic regression at 0.5: accuracy {accuracy:.3f}; caught {cm['tp']} of {cm['tp'] + cm['fn']} delays "
      f"(recall {cm['tp'] / (cm['tp'] + cm['fn']):.2f}); {cm['fp']} false alarms")

### Step 4 · WRITE

**WRITE · your interpretation**. The model's accuracy is only a few points above always guessing "no delay". What does the confusion
matrix show that the accuracy hides?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Bake-off

**Specification.** Compare six model families by cross-validated AUC, on training rows only.

### Step 5 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
pipelines = {name: make_model(name, build_preprocessor(), seed=SEED)
             for name in MODEL_NAMES}
auc_table, oof = cv_compare(pipelines, X_train, y_train, folds=5, seed=SEED)
print(auc_table.sort_values("cv_auc", ascending=False).round(3))

fig, ax = plt.subplots(figsize=(5, 4))
for name in MODEL_NAMES:
    fpr, tpr, _ = roc_curve(y_train, oof[name])
    ax.plot(fpr, tpr, label=f"{name} ({auc_table.loc[name, 'cv_auc']:.2f})")
ax.plot([0, 1], [0, 1], linestyle=":", color="grey")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate (recall)")
ax.legend(fontsize=7)
plt.show()

### Step 6 · WRITE

**WRITE · your interpretation**. The AUCs sit close together, around 0.66 to 0.71. What does AUC measure, and why is it not yet enough
to choose a model for this job?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: Cost function

**Specification.** Write the operational cost of a set of predictions. A missed maintenance delay disrupts a mission and
costs three times as much as an unneeded pre-departure inspection.

### Step 7 · DECIDE

In [ ]:
# DECIDE · DP-L06-1 · what a missed delay costs, against a false alarm that costs 1.
# Replace ____ with your choice. The choices are taught on the slide "The threshold is a decision".
# Read the specification above, then set the cost of a missed delay. A false alarm costs 1.
COST_MISSED, COST_FALSE_ALARM = ____, 1


def total_cost(y_true, y_pred):
    """Operational cost of yes/no predictions: missed delays and false alarms, each at its own cost."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    missed = int(np.sum((y_true == 1) & (y_pred == 0)))        # delays we did not flag
    false_alarms = int(np.sum((y_true == 0) & (y_pred == 1)))  # inspections that found nothing
    return COST_MISSED * missed + COST_FALSE_ALARM * false_alarms

### Step 8 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
flag_all = total_cost(y_train, np.ones(len(y_train)))
flag_none = total_cost(y_train, np.zeros(len(y_train)))
logit_05 = total_cost(y_train, (oof["logistic regression"] >= 0.5).astype(int))
print(f"inspect every departure: {flag_all}; inspect none: {flag_none}; logistic at 0.5: {logit_05}")

### Step 9 · WRITE

**WRITE · your interpretation**. Why is inspecting every departure cheaper here than the logistic model at 0.5?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 4: Operating point

**Specification.** For each model, find the threshold with the lowest cost on its out-of-fold predictions. Choose the
model and its threshold, and compare with the threshold that maximises accuracy.

### Step 10 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
curves = {name: cost_curve(y_train, oof[name], cost_fn=COST_MISSED, cost_fp=COST_FALSE_ALARM)
          for name in MODEL_NAMES}
summary = pd.DataFrame({name: {"lowest cost": c["cost"].min(),
                               "at threshold": pick_threshold(c, by="cost")}
                        for name, c in curves.items()}).T.sort_values("lowest cost")
print(summary.round(3))
chosen = summary.index[0]
curve = curves[chosen]
print("chosen model:", chosen)

### Step 11 · PREDICT

**PREDICT · before you run the next cell**. Will the cost-optimal threshold be above or below 0.5, and will its accuracy be higher or lower than
the accuracy-optimal threshold's?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 12 · DECIDE

In [ ]:
# DECIDE · DP-L06-2 · what the operating point is chosen against.
# Replace ____ with your choice. The choices are taught on the slide "Choosing the operating point".
# Choose the threshold for the chosen model: the one with the lowest cost, or the one with the highest accuracy.
best_threshold = pick_threshold(curve, by="____")
accuracy_threshold = pick_threshold(curve, by="accuracy")
for label, t in [("cost-optimal", best_threshold), ("accuracy-optimal", accuracy_threshold)]:
    row = curve.loc[(curve["threshold"] - t).abs().idxmin()]
    print(f"{label:<17} threshold {t:.3f}: cost {int(row['cost'])}, accuracy {row['accuracy']:.3f}, "
          f"missed {int(row['missed'])}, false alarms {int(row['false_alarms'])}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(curve["threshold"], curve["cost"], label="cost")
ax.axvline(best_threshold, linestyle="--", label="cost-optimal")
ax.axvline(accuracy_threshold, linestyle=":", label="accuracy-optimal")
ax.axhline(flag_all, color="grey", linewidth=0.8, label="inspect everything")
ax.set_xlabel("threshold on predicted delay risk")
ax.set_ylabel("cost (training, out of fold)")
ax.legend(fontsize=8)
plt.show()

### Step 13 · WRITE

**WRITE · your interpretation**. Explain why the cost-optimal threshold is "less accurate" and still the right choice.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 5: Test evaluation

**Specification.** Refit the chosen model on all training rows, score the untouched test set once, and read the result.

### Step 14 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
final = make_model(chosen, build_preprocessor(), seed=SEED).fit(X_train, y_train)
test_scores = final.predict_proba(X_test)[:, 1]
train_auc = roc_auc_score(y_train, final.predict_proba(X_train)[:, 1])
test_auc = roc_auc_score(y_test, test_scores)
print(f"{chosen}: AUC on training rows {train_auc:.3f}, cross-validated {auc_table.loc[chosen, 'cv_auc']:.3f}, "
      f"test {test_auc:.3f}")
for label, t in [("cost-optimal", best_threshold), ("accuracy-optimal", accuracy_threshold)]:
    pred = (test_scores >= t).astype(int)
    print(f"test at {label} threshold {t:.3f}: cost {total_cost(y_test, pred)}, "
          f"accuracy {(pred == y_test).mean():.3f}; {confusion_at(y_test, test_scores, t)}")
print(f"test, inspect everything: cost {total_cost(y_test, np.ones(len(y_test)))}")

### Step 15 · WRITE

**WRITE · your interpretation**. Compare the training, cross-validated and test AUC. What do they say about bias and variance, and
what is your recommendation to the group?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 16 · RUN

In [ ]:
# RUN · saves this part's output to work/.
choice = {"model": chosen, "threshold": round(float(best_threshold), 4), "cost_missed": COST_MISSED,
          "cost_false_alarm": COST_FALSE_ALARM,
          "cv_auc": round(float(auc_table.loc[chosen, "cv_auc"]), 4), "test_auc": round(float(test_auc), 4),
          "cv_auc_all": auc_table["cv_auc"].round(4).to_dict()}
(WORK / "L06_choice.json").write_text(json.dumps(choice, indent=2), encoding="utf-8")
pd.DataFrame({"mission_id": test["mission_id"], "score": test_scores.round(5)}).to_csv(
    WORK / "L06_test_scores.csv", index=False, encoding="utf-8")
print("saved work/L06_choice.json and work/L06_test_scores.csv")